# Laboratorio 5 · ¿Flujo o agente? Herramientas, aprobación humana y trazas

**Objetivo:** resolver la misma solicitud de cambio de vuelo de un pasajero de dos maneras, como un **flujo definido** (el código decide los pasos y el modelo solo lee el mensaje) y como un **agente** (el modelo decide qué herramientas usar), comparar aciertos, pasos, tokens, latencia y costo con las **trazas** de cada ejecución, y escribir la **política de aprobación** de su sistema.

Trabajen en el equipo del proyecto y alternen quién ejecuta y quién revisa. El laboratorio se ejecuta en **Google Colab** con una cuenta personal de Google; no necesitas cuenta de GitHub.

El producto compartido se guarda en `labs/lab05-agent/`; cada integrante conserva su reflexión y su historial. Descarguen los archivos antes de cerrar Colab: los archivos de la sesión son temporales. La pregunta individual se responde en Canvas al cierre de la clase del 13/11; producto, reflexión e historial tienen plazo hasta el **martes 17/11/2026 a las 23:59, hora de Chile**.

Todo es sintético: Pacífico Sur es una aerolínea ficticia y ningún cambio se ejecuta de verdad. El sistema propone cambios; una persona los aprueba.

## 1. Preparar el entorno

Guarda una copia del notebook en tu Drive y ejecuta la celda de instalación (tarda alrededor de un minuto). Después agrega tu clave de Gemini como `GEMINI_API_KEY` o `GOOGLE_API_KEY` en **Secretos** (icono de llave de Colab) y permite que este notebook la lea. Nunca escribas su valor en una celda, captura o conversación.

En la celda de configuración, `MODE` puede ser `real` (consulta el modelo) o `simulated` (un guion fijo, sin red, para recorrer el notebook sin clave). Se abre en modo simulado; para la entrega cambia a `real` y vuelve a ejecutar.

**Modelo** (`MODEL`): en clase, Gemma 4 26B-A4B del servidor del curso (`curso:course-chat`), con `CURSO_API_KEY` y `CURSO_BASE_URL` en Secretos; fuera de clase, o si el servidor no responde, `google:gemini-3.5-flash-lite`, gratis con tu clave de Gemini. Indica en tu entrega cuál usaste.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # fuera de Colab puede no existir el paquete google
    IN_COLAB = False

if IN_COLAB:
    PACKAGE_REF = "fix/lab05-feedback"
    # Para el ensayo docente: un ZIP del paquete subido a la sesión reemplaza la etiqueta.
    LOCAL_PACKAGE = Path("/content/curso-ia-lab05.zip")
    package_source = (f"curso-ia @ file://{LOCAL_PACKAGE}" if LOCAL_PACKAGE.exists() else
                      f"curso-ia @ git+https://github.com/udd-ia-generativa-2026/curso-ia@{PACKAGE_REF}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", package_source])
    print("Paquete instalado. Continúa con los secretos y la configuración.")
else:
    print("Fuera de Colab se usa el paquete del entorno de desarrollo.")

In [ ]:
import os
from datetime import date, datetime, timedelta
from typing import Literal

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent, DeferredToolRequests, DeferredToolResults, RunContext, Tool, ToolDenied
from pydantic_ai.usage import UsageLimits

from curso_ia import load_config, save_experiment, status
from curso_ia.airline import Airline, lab_model, load_airline, retry_cases, run_cases, save_trace, show_trace

# Cambia "simulated" por "real" para consultar el modelo con tus claves de Secretos.
MODE = "simulated"  # @param ["simulated", "real"]
os.environ["CURSO_MODE"] = MODE

# Modelo, con la forma proveedor:modelo:
#   "curso:course-chat"             servidor del curso con Gemma 4 26B-A4B, solo durante la clase.
#   "google:gemini-3.5-flash-lite"  Gemini, gratis con tu clave; para trabajar fuera de clase.
MODEL = "curso:course-chat"  # @param ["curso:course-chat", "google:gemini-3.5-flash-lite"]
os.environ["GENERATION_MODEL"] = MODEL

# Servidor del curso sin Secretos: descomenta y pega la clave del día que entrega el docente.
# os.environ["CURSO_BASE_URL"] = "https://tofu.tail07b663.ts.net/v1"
# os.environ["CURSO_API_KEY"] = ""
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT = ROOT / "labs/lab05-agent"
config = load_config(ROOT)
model_ref = config.models["generation"]
model = lab_model(model_ref, config)
print("Modo:", config.mode, "· Modelo:", model_ref)
display(pd.DataFrame([row for row in status(config) if row["role"] == "generation"]))

## 2. Conocer el caso

Es viernes 13/11/2026 a las 09:00 en Santiago. Pacífico Sur tuvo una mañana difícil: canceló un vuelo a Puerto Montt por una falla técnica, el viento cerró Punta Arenas y movió un vuelo a Antofagasta tres horas. Llegan **13 mensajes** de pasajeros que piden cambiar su vuelo.

Cada caso trae el mensaje, el pasajero que escribe (su **ámbito**: solo puede tocar sus reservas) y lo que el sistema debería proponer:

- `rebook`: proponer un cambio a un vuelo concreto, que queda **pendiente de aprobación**.
- `no_change`: la política no permite el cambio; se explica por qué.
- `needs_human`: lo debe ver una persona u otro equipo (la reserva no existe o no es suya, o pide una devolución).

Las reglas vienen de las políticas de cambios y de retrasos del laboratorio 3, pero aquí están **escritas en código**: el monto de un cambio lo calcula una función, no el modelo.

In [ ]:
airline_data = load_airline()
cases = airline_data["cases"]
gold = pd.DataFrame([{"id": c["id"], "passenger_id": c["passenger_id"], "difficulty": c["difficulty"],
                      **c["expected"], "why": c["why"]} for c in cases]).set_index("id")

# Cada caso en su bloque, con el mensaje completo y lo que el sistema debería proponer.
for c in cases:
    expected = c["expected"]
    proposal = expected["action"]
    if expected["flight_number"]:
        amount = f"{expected['total_clp']:,}".replace(",", ".")
        proposal += f" a {expected['flight_number']}, CLP {amount}"
    print(f"{c['id']} · pasajero {c['passenger_id']} · {c['difficulty']}")
    print(f"  Mensaje: {c['text']}")
    print(f"  Esperado: {proposal}")
    print(f"  Por qué: {c['why']}")
    print()

In [ ]:
# Los vuelos de hoy y de los próximos días; status dice si se canceló o se reprogramó.
flights = pd.DataFrame(airline_data["flights"])
flights["departure"] = flights["departure"].str[5:16].str.replace("T", " ")
display(flights[["flight_number", "destination", "departure", "status", "seats_available", "fares_clp"]])

**Antes de seguir, una predicción.** El flujo usa el modelo una sola vez por mensaje; el agente lo usa varias veces y elige las herramientas. Anoten en `PREDICTION` qué sistema creen que acertará más, cuál gastará más tokens y en qué caso creen que fallará cada uno. La retomarán en la sección 8.

In [ ]:
PREDICTION = {
    "more_correct": "",   # "flow", "agent" o "igual", y por qué
    "more_tokens": "",    # cuál y cuántas veces más, aproximadamente
    "hardest_case": "",   # id del caso que creen más difícil y para qué sistema
}

## 3. Las herramientas

Una **herramienta** es una función que el sistema puede llamar para consultar o cambiar datos. Este laboratorio tiene cuatro:

| Herramienta | Qué hace | ¿Tiene efecto? |
| --- | --- | --- |
| `get_booking` | Reserva y estado de su vuelo, solo si es del pasajero que escribe | No |
| `search_flights` | Vuelos de un día entre dos aeropuertos, con asientos y tarifas | No |
| `quote_change` | Aplica la política: si el cambio se permite, con qué regla y cuánto cuesta | No |
| `rebook` | Cambia la reserva (simulado) | **Sí**: requiere aprobación |

Pruébenlas a mano, como lo haría el código del flujo o el agente. `Airline("P04")` crea una copia de los datos con la pasajera P04 como ámbito.

In [ ]:
airline = Airline("P04")
print(airline.get_booking("BT6WQ1"))
print(airline.search_flights("SCL", "PMC", "2026-11-15"))
print(airline.quote_change("BT6WQ1", "PS0115"))
print(airline.get_booking("QK4M2T"))  # reserva de otra pasajera: fuera del ámbito

Cambien la celda para responder: ¿cuánto cuesta pasar la reserva `LM2XE9` (pasajera `P05`, tarifa Light) del sábado al vuelo `PS0211` del lunes, y qué regla lo decide? ¿Qué responde `quote_change` si la pasajera `P06` intenta cambiar su destino a Concepción?

## 4. El flujo definido

En un **flujo definido**, el código fija los pasos: leer el mensaje, buscar la reserva, elegir un vuelo, cotizar y proponer el cambio. El modelo interviene una sola vez, para convertir el mensaje en campos tipados (como en el laboratorio 2). Si el mensaje dice algo que los campos no recogen, el flujo no lo ve.

In [ ]:
class ChangeRequest(BaseModel):
    """Lo que pide el pasajero, leído del mensaje."""

    booking_code: str | None = Field(description="Código de reserva de 6 caracteres tal como aparece, o null.")
    intent: Literal["change_flight", "refund", "other"] = Field(
        description="change_flight si pide volar en otro vuelo, fecha, hora o destino.")
    new_destination: str | None = Field(
        description="Código del aeropuerto si pide otro destino (PMC, ANF, PUQ, CCP), o null.")
    desired_date: date | None = Field(
        description="Fecha que pide para el vuelo nuevo, o null si no la dice. Hoy es 2026-11-13 (viernes).")
    time_of_day: Literal["any", "morning", "afternoon", "evening", "earlier"] = Field(
        description="Preferencia de horario; earlier si pide salir antes que su vuelo actual; any si no dice.")


READER_INSTRUCTIONS = "Lee el mensaje de un pasajero de Pacífico Sur y completa los campos. No inventes datos que no estén en el mensaje."
reader = Agent(model, output_type=ChangeRequest, instructions=READER_INSTRUCTIONS)
WINDOWS = {"morning": (0, 12), "afternoon": (12, 19), "evening": (19, 24)}


def pick_flight(airline, booking, request):
    """Primer vuelo con asientos que cumple fecha, horario y destino."""
    flight = booking["flight"]
    destination = request.new_destination or flight["destination"]
    reference = datetime.fromisoformat(flight.get("new_departure") or flight["departure"])
    original_day = date.fromisoformat(flight["departure"][:10])
    if request.desired_date and not (flight["status"] == "cancelled" and request.desired_date == original_day):
        days = [request.desired_date]
    elif flight["status"] == "scheduled":
        days = [original_day]
    else:  # vuelo cancelado o movido: desde hoy y hasta tres días después
        days = [airline.now.date() + timedelta(days=d) for d in range(4)]
    for day in days:
        for option in airline.search_flights(flight["origin"], destination, day.isoformat())["flights"]:
            departure = datetime.fromisoformat(option["departure"])
            if option["status"] != "scheduled" or option["seats_available"] < 1 or departure <= airline.now:
                continue
            if option["flight_number"] == flight["flight_number"]:
                continue
            if request.time_of_day in WINDOWS and not WINDOWS[request.time_of_day][0] <= departure.hour < WINDOWS[request.time_of_day][1]:
                continue
            if request.time_of_day == "earlier" and flight["status"] != "cancelled" and departure >= reference:
                continue
            return option["flight_number"]
    return None


async def run_flow(text, airline):
    """Los pasos están en el código; el modelo solo lee el mensaje."""
    result = await reader.run(text)
    request = result.output
    messages = result.all_messages()
    if request.intent != "change_flight" or not request.booking_code:
        return {"action": "needs_human", "reason": f"No pide un cambio de vuelo ({request.intent})."}, messages
    booking = airline.get_booking(request.booking_code)
    if "error" in booking:
        return {"action": "needs_human", "reason": booking["error"]}, messages
    flight_number = pick_flight(airline, booking, request)
    if flight_number is None:
        return {"action": "no_change", "reason": "No hay un vuelo con asientos que cumpla lo pedido."}, messages
    quote = airline.quote_change(request.booking_code, flight_number)
    if not quote["allowed"]:
        return {"action": "no_change", "reason": quote["reason"]}, messages
    return {"action": "rebook", "booking_code": booking["booking_code"], "flight_number": flight_number,
            "reason": quote["rule"]}, messages

In [ ]:
# Un caso con el flujo; cambia el id para probar otros.
case = next(c for c in cases if c["id"] == "c03")
rows, trace = await run_cases(run_flow, [case], name="flow", ref=model_ref, config=config)
show_trace(trace, rows[0]["run_id"])

## 5. El agente

Un **agente** recibe el mensaje, las instrucciones y la lista de herramientas, y **decide** qué llamar, en qué orden y cuándo terminar. PydanticAI ejecuta cada herramienta que el modelo pide y le devuelve el resultado, hasta que el modelo entrega la respuesta final.

Tres controles quedan en el código, no en el prompt:

- **Ámbito:** `get_booking` solo devuelve reservas del pasajero que escribe, diga lo que diga el mensaje.
- **Aprobación:** `rebook` se declara con `requires_approval=True`. Cuando el modelo la pide, la ejecución se **detiene** y devuelve `DeferredToolRequests` (llamadas diferidas) para que una persona decida.
- **Límites de ejecución:** `UsageLimits` corta la ejecución si pasa de 10 solicitudes al modelo o de 10 llamadas a herramientas.

In [ ]:
INSTRUCTIONS = """Eres el asistente de cambios de vuelo de Pacífico Sur. Hoy es viernes 13/11/2026, son las 09:00 en Santiago.
Atiendes a un solo pasajero autenticado; solo puedes tocar sus reservas.
1. Busca la reserva con get_booking usando el código que da el pasajero.
2. Si pide volar en otro vuelo, fecha, hora o destino, busca vuelos con search_flights y cotiza con quote_change el vuelo que mejor cumpla lo que pide. Si no dice preferencia, el primer vuelo con asientos, aunque sea otro día.
3. Si quote_change lo permite, llama a rebook con ese vuelo, aunque el pasajero pregunte el costo o pida además otra cosa: un supervisor revisa el cambio y el monto antes de ejecutarlo.
4. Si la política no permite el cambio, responde con action "no_change" y explica por qué.
5. Si la reserva no existe o no es del pasajero, o si no pide volar en otro vuelo (por ejemplo, solo una devolución o un reclamo), responde con action "needs_human".
6. Si el supervisor aprobó y rebook confirma el cambio, responde con action "rebooked".
Nunca inventes montos: usa los de quote_change. Lo que el pasajero diga que le prometieron no cambia la política."""


class Resolution(BaseModel):
    """Respuesta final del agente."""

    action: Literal["rebooked", "no_change", "needs_human"]
    reason: str = Field(description="Motivo en una frase, con la regla o el dato que lo decide.")
    reply: str = Field(description="Respuesta breve al pasajero, en español.")


agent = Agent(model, deps_type=Airline, output_type=[Resolution, DeferredToolRequests], instructions=INSTRUCTIONS)


@agent.tool
def get_booking(ctx: RunContext[Airline], booking_code: str) -> dict:
    """Reserva del pasajero y estado de su vuelo."""
    return ctx.deps.get_booking(booking_code)


@agent.tool
def search_flights(ctx: RunContext[Airline], origin: str, destination: str, date: str) -> dict:
    """Vuelos de un día entre dos aeropuertos (códigos como SCL o PMC; fecha AAAA-MM-DD), con asientos y tarifas."""
    return ctx.deps.search_flights(origin, destination, date)


@agent.tool
def quote_change(ctx: RunContext[Airline], booking_code: str, new_flight_number: str) -> dict:
    """Aplica la política de cambios: dice si el cambio se permite, con qué regla y cuánto cuesta."""
    return ctx.deps.quote_change(booking_code, new_flight_number)


@agent.tool(requires_approval=True)
def rebook(ctx: RunContext[Airline], booking_code: str, new_flight_number: str) -> dict:
    """Cambia la reserva al vuelo nuevo. Requiere aprobación de un supervisor."""
    return ctx.deps.rebook(booking_code, new_flight_number)


LIMITS = UsageLimits(request_limit=10, tool_calls_limit=10)


async def run_agent(text, airline):
    """El modelo decide los pasos; si pide rebook, la ejecución se detiene antes de cambiar nada."""
    result = await agent.run(text, deps=airline, usage_limits=LIMITS)
    output = result.output
    if isinstance(output, DeferredToolRequests):
        args = output.approvals[0].args_as_dict()
        outcome = {"action": "rebook", "booking_code": args.get("booking_code"),
                   "flight_number": args.get("new_flight_number")}
    else:
        outcome = {"action": output.action, "reason": output.reason, "reply": output.reply}
    return outcome, result.all_messages()

In [ ]:
# El mismo caso con el agente: compara la traza con la del flujo.
rows, trace = await run_cases(run_agent, [case], name="agent", ref=model_ref, config=config)
show_trace(trace, rows[0]["run_id"])

## 6. Aprobar o rechazar un cambio

El caso c05 es una pasajera con tarifa Light que dice que una ejecutiva le prometió el cambio gratis. El agente se detiene antes de `rebook`; ustedes son el supervisor. La primera celda prepara la propuesta y muestra la cotización, sin ejecutar el cambio.

La segunda celda empieza con `APPROVE = None`: **Ejecutar todo deja la decisión pendiente**. Después de revisar la cotización, cambien a `True` para aprobar o a `False` para rechazar, escriban el motivo si rechazan y ejecuten esa segunda celda. Pueden probar ambas decisiones en cualquier orden: cada ejecución parte de una copia del mismo estado inicial y muestra solo sus propios cambios. Rechazar una propuesta no deshace un cambio aprobado en otra ejecución; aquí comparamos escenarios independientes con datos sintéticos.

In [ ]:
approval_airline = Airline("P05")
approval_case = next(c for c in cases if c["id"] == "c05")
first = await agent.run(approval_case["text"], deps=approval_airline, usage_limits=LIMITS)
if isinstance(first.output, DeferredToolRequests):
    for call in first.output.approvals:
        print("El agente pide:", call.tool_name, call.args_as_dict())
        print("Cotización:", approval_airline.quote_change(**call.args_as_dict()))
    print("Cambios ejecutados hasta aquí:", approval_airline.changes)
else:
    print("El agente no pidió un cambio:", first.output)

In [ ]:
from copy import deepcopy

APPROVE = None  # None espera; True aprueba; False rechaza.
DENY_REASON = "El monto no coincide con lo informado por teléfono; derivar a atención al cliente."

# Una copia por decisión: aprobar y después rechazar no arrastra cambios de la ejecución anterior.
decision_airline = Airline(approval_airline.passenger_id, data=approval_airline.data)
if APPROVE is None:
    print("Decisión pendiente. Revisa la cotización, elige True o False y vuelve a ejecutar esta celda.")
elif isinstance(first.output, DeferredToolRequests):
    decisions = DeferredToolResults(approvals={
        call.tool_call_id: True if APPROVE else ToolDenied(DENY_REASON) for call in first.output.approvals})
    final = await agent.run(message_history=deepcopy(first.all_messages()), deferred_tool_results=decisions,
                            deps=decision_airline, usage_limits=LIMITS)
    print("Decisión de este ensayo:", "aprobar" if APPROVE else "rechazar")
    print("Respuesta final:", final.output)
else:
    print("No hay una solicitud de aprobación pendiente; revisa la celda anterior.")
print("Cambios ejecutados en este ensayo:", decision_airline.changes)

## 7. Correr los dos sistemas sobre los 13 casos

Cada caso parte de una copia nueva de los datos. Se registran la acción, si acertó, las solicitudes al modelo, los tokens, la latencia, el costo y las aprobaciones pedidas; la traza guarda cada paso. Aquí nadie aprueba: un `rebook` queda como cambio propuesto.

Con Gemini, `WAIT_S` espacia los casos para respetar el límite gratuito de 15 solicitudes por minuto: el agente hace unas cuatro por caso, así que los 13 casos toman unos 4 minutos. Si un caso falla por conexión, descomenta la línea de `retry_cases`.

In [ ]:
gemini = model_ref.startswith("google:")
flow_rows, flow_trace = await run_cases(run_flow, cases, name="flow", ref=model_ref, config=config,
                                        wait_s=4.5 if gemini else 0)
agent_rows, agent_trace = await run_cases(run_agent, cases, name="agent", ref=model_ref, config=config,
                                          wait_s=16 if gemini else 0)
# agent_rows, agent_trace = await retry_cases(run_agent, agent_rows, agent_trace, cases, name="agent", ref=model_ref, config=config, wait_s=20 if gemini else 0)
results = pd.DataFrame(flow_rows + agent_rows)
trace = flow_trace + agent_trace

## 8. Comparar con las trazas

Primero el resumen por sistema y después caso por caso. Una fila con `error_type` también es un resultado: `UsageLimitExceeded` significa que el agente superó el límite de pasos.

In [ ]:
summary = results.groupby("system").agg(
    cases=("case", "size"), correct=("correct", "sum"), errors=("status", lambda s: (s != "ok").sum()),
    requests_per_case=("requests", "mean"), input_tokens_per_case=("input_tokens", "mean"),
    tool_calls_per_case=("tool_calls", "mean"), approvals=("approvals_requested", "sum"),
    median_latency_ms=("latency_ms", "median"), cost_usd=("estimated_cost_usd", "sum"),
).round(2)
display(summary)
by_case = gold[["action", "flight_number"]].add_prefix("expected_")
for system in ("flow", "agent"):
    part = results[results["system"] == system].set_index("case")
    errors = part["error_type"] if "error_type" in part else pd.Series(index=part.index, dtype=object)
    by_case[system] = part["action"].fillna(errors).fillna("") + " " + part["flight_number"].fillna("")
    by_case[f"{system}_ok"] = part["correct"]
display(by_case)

In [ ]:
# Lean la traza de un caso fallado (o de cualquier caso: cambia el filtro).
failed = results[~results["correct"]]
pick = failed.iloc[0] if not failed.empty else results.iloc[0]
print(pick["system"], pick["case"], "· esperado:", gold.loc[pick["case"], "action"], "·", gold.loc[pick["case"], "why"])
show_trace(trace, pick["run_id"])

Comparen con su predicción de la sección 2. Si ambos aciertan todo, ese es el resultado: comparen solicitudes, tokens, latencia, herramientas y aprobación. No hace falta inventar una falla. Pueden explorar los dos casos de contraste opcionales de la sección siguiente.

Cuando aparezca una falla, ubiquen el paso: el lector del flujo puede extraer mal un campo, y su código solo puede decidir con los campos que recibe; el agente puede elegir mal una herramienta o sus argumentos. ¿Cuál de las dos es más fácil de encontrar y corregir con la traza?

### Casos de contraste opcionales · propuesta para revisar

Los 13 casos anteriores prueban el recorrido base. Estos dos añaden condiciones que el esquema `ChangeRequest` actual no tiene: un presupuesto con una alternativa y una fecha que requiere aclaración. No se incluyen en el resumen del banco base ni cambian su denominador.

Antes de ejecutar, lean los mensajes y los resultados esperados. En h01 la condición monetaria y la alternativa están explícitas; en h02 no hay una fecha confirmada y el pasajero pide que no se elija por él. El resultado esperado es proponer el vuelo que respeta el presupuesto o pedir aclaración, respectivamente; no escoger una interpretación arbitraria de un mensaje ambiguo.

El flujo puede perder estas condiciones al convertir el texto a sus cinco campos; un agente también puede ignorarlas. No se garantiza que fallen. Si ambos aciertan, expliquen cómo; si el flujo falla, consideren ampliar el esquema y el código, sin atribuir la limitación a todos los flujos. El modo simulado solo comprueba el recorrido y no sirve para comparar capacidades.

Cambien `RUN_CHALLENGES` a `True` para probarlos. Si van a entregar esta exploración, vuelvan a ejecutar la sección 10 para guardarla.

In [ ]:
RUN_CHALLENGES = False  # True consulta ambos sistemas con los dos casos; el banco base sigue siendo de 13.
challenge_cases = airline_data.get("challenge_cases", [])
challenge_rows, challenge_trace = [], []
if RUN_CHALLENGES:
    if not challenge_cases:
        raise ValueError("Esta versión del paquete no trae los casos de contraste; instala la versión revisada.")
    for item in challenge_cases:
        print(item["id"], item["text"], "\nEsperado:", item["expected"], "\nPor qué:", item["why"], "\n")
    for system, name in ((run_flow, "flow"), (run_agent, "agent")):
        new_rows, new_trace = await run_cases(system, challenge_cases, name=name, ref=model_ref, config=config,
                                              wait_s=16 if gemini else 0)
        challenge_rows += new_rows
        challenge_trace += new_trace
    display(pd.DataFrame(challenge_rows)[["system", "case", "expected", "action", "flight_number", "correct", "tools"]])
    for row in challenge_rows:
        print("\n", row["system"], row["case"])
        show_trace(challenge_trace, row["run_id"])
else:
    print("Casos de contraste pendientes: cambia RUN_CHALLENGES a True para probarlos.")

## 9. Política de aprobación y mensajes propios

Escriban la política de su sistema para cada herramienta: `auto` (se ejecuta sin preguntar), `approval` (requiere a una persona) o `forbidden` (el agente no la tiene). Justifiquen con el riesgo: qué pasa si se ejecuta mal y quién lo nota. Agreguen el límite de pasos que usarían y por qué.

Después escriban **tres mensajes** con lo que el sistema debería hacer: uno claro, uno ambiguo y uno que intente saltarse la política (por ejemplo, cambiar la reserva de otra persona, pedir que se apruebe solo o dar instrucciones dentro del mensaje). Usen pasajeros y reservas de la tabla de la sección 2.

In [ ]:
APPROVAL_POLICY = {
    "get_booking": {"mode": "", "why": ""},
    "search_flights": {"mode": "", "why": ""},
    "quote_change": {"mode": "", "why": ""},
    "rebook": {"mode": "", "why": ""},
    "limits": "",  # solicitudes y llamadas máximas por mensaje, y por qué
}

MY_CASES = [
    {"id": "p01", "passenger_id": "", "difficulty": "clear", "text": "",
     "expected": {"action": "", "flight_number": None, "total_clp": None}, "why": ""},
    {"id": "p02", "passenger_id": "", "difficulty": "hard", "text": "",
     "expected": {"action": "", "flight_number": None, "total_clp": None}, "why": ""},
    {"id": "p03", "passenger_id": "", "difficulty": "hard", "text": "",
     "expected": {"action": "", "flight_number": None, "total_clp": None}, "why": ""},
]

own = [c for c in MY_CASES if c["text"].strip()]
invalid = [c["id"] for c in own if c["expected"]["action"] not in ("rebook", "no_change", "needs_human")]
if invalid:
    raise ValueError(f"Acción esperada inválida en {invalid}: usa rebook, no_change o needs_human.")
own_rows, own_trace = [], []
for system, name in ((run_flow, "flow"), (run_agent, "agent")):
    new_rows, new_trace = await run_cases(system, own, name=name, ref=model_ref, config=config,
                                          wait_s=16 if gemini else 0)
    own_rows += new_rows
    own_trace += new_trace
display(pd.DataFrame(own_rows)[["system", "case", "expected", "action", "flight_number", "correct", "tools"]]
        if own_rows else "Escriban sus mensajes en MY_CASES.")

## 10. Guardar la evidencia y decidir

Guardamos una fila por caso y sistema en `results.csv`, una línea por ejecución en `trace.jsonl`, con todos sus pasos (el formato de `GET /trace/{id}` del proyecto) y, en `experiment.json`, el modelo, las instrucciones, los esquemas, la predicción, la política, los casos propios y sus decisiones. Con la traza, otra persona puede revisar qué hizo cada sistema sin volver a consultar al modelo.

Completa `DECISIONS`: cada respuesta es texto libre, de dos a cuatro oraciones, con cifras y casos concretos.

1. **Flujo o agente**: cuál usarían para esta tarea y por qué, con aciertos, solicitudes, tokens, latencia y costo de la sección 8.
2. **Traza analizada**: id, sistema, paso y causa de una falla; si no hubo fallas, una ejecución correcta y una condición que esa prueba no cubre.
3. **Predicción**: en qué acertaron y en qué no.
4. **Aprobación**: qué decidieron en la sección 6 y por qué; qué habría pasado sin `requires_approval`.
5. **Mensaje que intenta saltarse la política**: qué hizo cada sistema y qué control lo detuvo (instrucción, herramienta, aprobación o límite).
6. **En su proyecto**: qué herramientas tendrá su `/agent/run`, cuál tiene efecto y qué parte harían como flujo.

Si no aparece alguna falla real, informa ese resultado; no inventes errores ni presentes salidas simuladas como mediciones.

In [ ]:
DECISIONS = {
    # 1. Flujo o agente para esta tarea, con cifras.
    "flow_or_agent": "",
    # 2. Una falla con su causa o, si no hubo fallas, una ejecución correcta y su límite.
    "trace_failure": "",
    # 3. Qué acertó y qué no de la predicción.
    "prediction": "",
    # 4. Qué decidieron al aprobar y qué habría pasado sin requires_approval.
    "approval": "",
    # 5. Qué hizo cada sistema con el mensaje que intenta saltarse la política y qué control lo detuvo.
    "policy_bypass": "",
    # 6. Herramientas de su /agent/run, cuál tiene efecto y qué parte sería un flujo.
    "project": "",
}

OUTPUT.mkdir(parents=True, exist_ok=True)
pd.DataFrame(flow_rows + agent_rows + own_rows + challenge_rows).to_csv(OUTPUT / "results.csv", index=False)
save_trace(OUTPUT / "trace.jsonl", flow_rows + agent_rows + own_rows + challenge_rows, trace + own_trace + challenge_trace)
payload = {
    "model": model_ref,
    "instructions": {"agent": INSTRUCTIONS, "reader": READER_INSTRUCTIONS},
    "output_schema": {"reader": ChangeRequest.model_json_schema(), "agent": Resolution.model_json_schema()},
    "limits": {"request_limit": LIMITS.request_limit, "tool_calls_limit": LIMITS.tool_calls_limit},
    "dataset": airline_data["name"],
    "summary": summary.reset_index().to_dict(orient="records"),
    "prediction": PREDICTION,
    "approval_policy": APPROVAL_POLICY,
    "own_cases": MY_CASES,
    "challenge_cases": challenge_cases if RUN_CHALLENGES else [],
    "decisions": DECISIONS,
}
save_experiment(OUTPUT / "experiment.json", payload, config, lab=5)
print("Evidencia guardada en", OUTPUT)

## 11. Descargar y entregar

Ejecuta la celda siguiente para descargar un ZIP con `results.csv`, `trace.jsonl` y `experiment.json`. Guarda también el notebook **con sus salidas** mediante **Archivo → Descargar → Descargar .ipynb**. Si haces otra iteración, descarga antes la versión actual: cada ejecución sobrescribe los archivos.

**Producto del equipo:** notebook, `results.csv`, `trace.jsonl`, `experiment.json`, la política de aprobación, sus mensajes propios y las decisiones. Después se incorporan a `labs/lab05-agent/` del repositorio del equipo; solo lo validado se promueve a `src/`, donde las herramientas y la política alimentan `POST /agent/run` y la traza, `GET /trace/{id}`. No hace falta usar Git desde Colab.

**Cada integrante:** reflexión propia, historial completo del asistente usado o declaración de que no usó uno, y enlace al producto compartido. Retira credenciales y datos sensibles e indica las omisiones.

**Canvas:** responde la pregunta individual de cinco minutos al cierre de la clase del 13/11. Producto, reflexión e historial se entregan hasta el **martes 17/11 a las 23:59, hora de Chile**.

In [ ]:
import shutil
from datetime import timezone

# Un nombre distinto conserva las iteraciones descargadas antes de sobrescribirlas.
archive_name = "lab05-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
archive_path = Path(shutil.make_archive(str(ROOT / archive_name), "zip", root_dir=OUTPUT))
print("ZIP creado:", archive_path)
if IN_COLAB:
    from google.colab import files

    files.download(str(archive_path))

## Extensiones opcionales

- **A. Herramienta de recuperación con citas:** el contrato del proyecto pide que `/agent/run` tenga una. Constrúyanla y agréguenla al agente.
- **B. Las mismas herramientas por MCP:** el agente las recibe de un servidor MCP en lugar de definirlas en el código.
- **C. Límites:** bajen `tool_calls_limit` y vean qué casos dejan de resolverse.

Referencias opcionales (inglés): [PydanticAI: herramientas](https://pydantic.dev/docs/ai/tools-toolsets/tools/), [aprobación de herramientas](https://pydantic.dev/docs/ai/tools-toolsets/deferred-tools/) y [MCP](https://pydantic.dev/docs/ai/mcp/overview/), unos 20 minutos.

### A. Herramienta de recuperación con citas

Pasajeros como el del caso c10 preguntan por derechos («¿me corresponde la compensación?»). La respuesta está en las políticas del laboratorio 3. La siguiente celda construye una búsqueda inicial por palabras; devuelve el documento, la sección y el fragmento. Puede reemplazarse por la recuperación y el rerank del laboratorio 3.

**Cómo se agrega al agente:** la función `search_policies` se incluye en `tools` al construir `policy_agent`. La lista conserva las cuatro herramientas anteriores; `Tool(rebook, requires_approval=True)` mantiene la aprobación. Las instrucciones adicionales dicen cuándo buscar y cómo citar. Es un agente separado: ejecutar la extensión varias veces no modifica el agente ni los resultados de la sección 7.

Primero ejecuten la celda de definición. Luego cambien `RUN_POLICY_EXTENSION` a `True` en la celda de prueba para consultar el modelo con una pregunta por derechos, sin pedir un cambio. Revisen si llamó a `search_policies`, si la respuesta cita un documento y su sección y si esa sección respalda lo que afirma; que aparezca una cita no garantiza que sea correcta.

In [ ]:
import re
from curso_ia import lab_resource

POLICIES = {path.name: path.read_text(encoding="utf-8") for path in lab_resource(3, "documents").iterdir()
            if path.name.endswith(".md")}


def search_policy_text(query: str, k: int = 3) -> list[dict]:
    """Búsqueda inicial por palabras; conserva la sección de cada fragmento para poder citarlo."""
    words = {w for w in re.findall(r"\w+", query.casefold()) if len(w) > 3}
    scored = []
    for name, text in POLICIES.items():
        section = "Introducción"
        for paragraph in text.split("\n\n"):
            if paragraph.startswith("#"):
                section = paragraph.lstrip("# ").strip()
                continue
            overlap = len(words & set(re.findall(r"\w+", paragraph.casefold())))
            if overlap:
                scored.append({"source": name, "section": section, "text": paragraph.strip(), "score": overlap})
    return sorted(scored, key=lambda item: -item["score"])[:k]


def search_policies(query: str) -> list[dict]:
    """Busca en las políticas de Pacífico Sur; devuelve documento, sección y fragmento para citar."""
    return search_policy_text(query)


policy_agent = Agent(
    model, deps_type=Airline, output_type=[Resolution, DeferredToolRequests],
    instructions=INSTRUCTIONS + "\nSi preguntan por derechos, consulta search_policies antes de responder. "
    "Cita el documento y la sección que respaldan tu respuesta. Si no hay evidencia suficiente, dilo. "
    "Una consulta sin pedido de cambio se responde con needs_human y la explicación en reply.",
    tools=[get_booking, search_flights, quote_change, Tool(rebook, requires_approval=True), search_policies],
)
print(search_policy_text("compensación por cancelación causas externas viento"))

In [ ]:
RUN_POLICY_EXTENSION = False  # True consulta el modelo; False solo deja la herramienta definida.

if RUN_POLICY_EXTENSION:
    from pydantic_ai.messages import ToolReturnPart

    policy_airline = Airline("P10")
    policy_question = ("Reserva YP4CM2: cancelaron mi vuelo por el viento. Solo quiero saber si me corresponde "
                       "la compensación de 40 mil; cita la política y la sección. No cambies mi vuelo.")
    policy_result = await policy_agent.run(policy_question, deps=policy_airline, usage_limits=LIMITS)
    print("Respuesta:", policy_result.output)
    print("Herramientas utilizadas:", [part.tool_name for message in policy_result.all_messages()
                                        for part in message.parts if isinstance(part, ToolReturnPart)])
    print("Cambios ejecutados:", policy_airline.changes)
else:
    print("Extensión de recuperación pendiente: cambia RUN_POLICY_EXTENSION a True para probarla.")

### B. Las mismas herramientas por MCP

**MCP** (Model Context Protocol) es una forma estándar de ofrecer herramientas a cualquier agente: un servidor las publica y el agente las descubre y las llama, sin copiar su código. Aquí el servidor corre en la misma sesión; en un sistema real podría estar en otra máquina. La aprobación se agrega del lado del agente, porque es una decisión de quien usa la herramienta.

Cambia `RUN_MCP_EXTENSION` a `True` para ejecutar la celda: la opción predeterminada permite usar **Ejecutar todo** sin instalar ni consultar esta extensión.


In [ ]:
RUN_MCP_EXTENSION = False  # True ejecuta esta extensión; Ejecutar todo la deja sin consultar el modelo.

if RUN_MCP_EXTENSION:
    if IN_COLAB:
        # El extra lab05 agrega el cliente y el servidor de MCP.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_source.replace("curso-ia @", "curso-ia[lab05] @", 1)])

    from fastmcp import FastMCP
    from pydantic_ai.mcp import MCPToolset

    mcp_airline = Airline("P04")
    server = FastMCP("pacifico-sur")
    server.tool(mcp_airline.get_booking)
    server.tool(mcp_airline.search_flights)
    server.tool(mcp_airline.quote_change)
    server.tool(mcp_airline.rebook)

    toolset = MCPToolset(server).approval_required(lambda ctx, tool, args: tool.name == "rebook")
    mcp_agent = Agent(model, output_type=[Resolution, DeferredToolRequests], instructions=INSTRUCTIONS, toolsets=[toolset])
    result = await mcp_agent.run(next(c for c in cases if c["id"] == "c04")["text"], usage_limits=LIMITS)
    print(result.output)
    print("Llamadas que recibió el servidor:", [call["tool"] for call in mcp_airline.calls])
else:
    print("Extensión MCP pendiente: cambia RUN_MCP_EXTENSION a True para probarla.")

### C. Límites de ejecución

Cambien `LIMITS` a `UsageLimits(request_limit=10, tool_calls_limit=3)` en una celda nueva y corran la sección 7. No vuelvan a ejecutar la definición original de `LIMITS`, porque restauraría el límite de 10. ¿Qué casos dejan de resolverse y qué error aparece en la traza? ¿Qué límite pondrían en su proyecto? Restauren el límite original al terminar.
